In [ ]:
# Supprimer le dossier de cache pour NeuroDSL
cache_dir = joinpath(homedir(), ".julia", "compiled", "v1.10", "NeuroDSL")
if isdir(cache_dir)
    rm(cache_dir; recursive=true, force=true)
    println("Cache supprimé : $cache_dir")
else
    println("Pas de cache trouvé")
end

In [ ]:
include(raw"C:\Users\Nevermind\Desktop\NeuroDSL\src\NeuroDSL.jl")
using .NeuroDSL

In [ ]:
using NeuroDSL

# 1. ENREGISTRER LES OPÉRATEURS (obligatoire avant toute utilisation)
NeuroDSL.register_op!(:wsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) ->
        (@. out = attrs[:weights][1] * inputs[1] + attrs[:weights][2] * inputs[2])
)
NeuroDSL.register_op!(:nsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
        copyto!(out, inputs[1])
        for i in 2:length(inputs)
            out .+= inputs[i]
        end
    end
)
NeuroDSL.register_op!(:identity,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> copyto!(out, inputs[1])
)

# 2. Construction du graphe
g = NeuroGraph(namespace=:fib, device=Backend.CPUDevice())

builder = @neuro g ns=:fib begin
    @node x0 = 1.0
    @node x1 = 2.0
    @rule fib(n::Int) = n <= 1 ? Symbol(:x, n) : wsum(weights=[1.0, 1.0], fib(n-1), fib(n-2))
    @node f8 = fib(8)
    even_terms = [fib(i) for i in 0:2:8]
    @node sn = nsum(even_terms...)
end

# 3. Exécution forward
log = ExecutionLog()
demand!(g, :sn; ctx_store=CtxStore(), namespace=:fib, log=log)

# 4. Visualisation interactive
save_interactive_graph(g, log, "fib_stream.html"; title="Fibonacci Stream")

In [ ]:
using NeuroDSL

In [ ]:
# V2
using NeuroDSL

# Forme courte (recommandée) : attrs[:nom] doit être écrit explicitement
@defop wsum      out = attrs[:w1] * inputs[1] + attrs[:w2] * inputs[2]
@defop scale_add out = attrs[:factor] * inputs[1] + inputs[2]
@defop identity  out = inputs[1]

# Forme lambda explicite (pour des corps complexes)
@defop nsum (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
    for i in 2:length(inputs)
        out .+= inputs[i]
    end
end

In [ ]:
# V2
using NeuroDSL

using NeuroDSL

# Forme courte (recommandée) : attrs[:nom] doit être écrit explicitement
@defop wsum      out = attrs[:w1] * inputs[1] + attrs[:w2] * inputs[2]
@defop scale_add out = attrs[:factor] * inputs[1] + inputs[2]
@defop identity  out = inputs[1]

# Forme lambda explicite (pour des corps complexes)
@defop nsum (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
    for i in 2:length(inputs)
        out .+= inputs[i]
    end
end

# ── Construction du graphe ───────────────────────────────────────────
g = NeuroGraph(namespace=:fib, device=Backend.CPUDevice())

builder = @neuro g ns=:fib begin
    @node x0 = 1.0
    @node x1 = 2.0
    @rule fib(n::Int) = n <= 1 ? Symbol(:x, n) : wsum(w1=1.0, w2=1.0, fib(n-1), fib(n-2))
    @node f8 = fib(8)
    even_terms = [fib(i) for i in 0:2:8]
    @node sn = nsum(even_terms...)
end

# ── Exécution ────────────────────────────────────────────────────────
log = ExecutionLog()
demand!(g, :sn; ctx_store=CtxStore(), namespace=:fib, log=log)

# ── Visualisation ────────────────────────────────────────────────────
save_interactive_graph(g, log, "fib_stream.html"; title="Fibonacci Stream")

In [ ]:
using NeuroDSL

# ── 1. Enregistrement des opérateurs ─────────────────────────────────────
NeuroDSL.register_op!(:wsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) ->
        (@. out = attrs[:weights][1] * inputs[1] + attrs[:weights][2] * inputs[2])
)
NeuroDSL.register_op!(:nsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
        copyto!(out, inputs[1])
        for i in 2:length(inputs)
            out .+= inputs[i]
        end
    end
)
NeuroDSL.register_op!(:identity,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> copyto!(out, inputs[1])
)

# ── 2. Construction déclarative du graphe ────────────────────────────────
g = NeuroGraph(namespace=:vec, device=Backend.CPUDevice())

builder = @neuro g ns=:vec begin
    # Vecteurs initiaux (taille 2)
    @node v0 = [1.0, 0.0]
    @node v1 = [0.0, 1.0]

    # Règle récursive vectorielle : v_n = 0.5*v_{n-1} + 0.5*v_{n-2}
    @rule vec_seq(n::Int) = n <= 1 ? Symbol(:v, n) : wsum(weights=[0.5, 0.5], vec_seq(n-1), vec_seq(n-2))

    # Calculer les 5 premiers termes (v2 à v5)
    @node v2 = vec_seq(2)
    @node v3 = vec_seq(3)
    @node v4 = vec_seq(4)
    @node v5 = vec_seq(5)

    # Moyenne des 3 derniers vecteurs (v3, v4, v5)
    @node avg = nsum(v3, v4, v5)   # sera divisé par 3 dans un second temps ? ou utiliser un opérateur scale
end

# ── 3. Exécution forward ─────────────────────────────────────────────────
log = ExecutionLog()
demand!(g, :avg; ctx_store=CtxStore(), namespace=:vec, log=log)

# Afficher les valeurs finales
println("v0 = ", Array(g.nodes[:vec][:v0].value))
println("v1 = ", Array(g.nodes[:vec][:v1].value))
println("v5 = ", Array(g.nodes[:vec][:v5].value))
println("avg = ", Array(g.nodes[:vec][:avg].value))

# ── 4. Visualisation interactive ─────────────────────────────────────────
save_interactive_graph(g, log, "vec_stream.html"; title="Vector Stream")

In [ ]:
# V2 vec stream
using NeuroDSL

# ── 1. Définition des opérateurs avec @defop ───────────────────────────
@defop wsum out = attrs[:weights][1] * inputs[1] + attrs[:weights][2] * inputs[2]

@defop nsum (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
    for i in 2:length(inputs)
        out .+= inputs[i]
    end
end

@defop identity out = inputs[1]

# ── 2. Construction déclarative du graphe ────────────────────────────────
g = NeuroGraph(namespace=:vec, device=Backend.CPUDevice())

builder = @neuro g ns=:vec begin
    # Vecteurs initiaux (taille 2)
    @node v0 = [1.0, 0.0]
    @node v1 = [0.0, 1.0]

    # Règle récursive vectorielle : v_n = 0.5*v_{n-1} + 0.5*v_{n-2}
    @rule vec_seq(n::Int) = n <= 1 ? Symbol(:v, n) : wsum(weights=[0.5, 0.5], vec_seq(n-1), vec_seq(n-2))

    # Calculer les 5 premiers termes (v2 à v5)
    @node v2 = vec_seq(2)
    @node v3 = vec_seq(3)
    @node v4 = vec_seq(4)
    @node v5 = vec_seq(5)

    # Moyenne des 3 derniers vecteurs (v3, v4, v5)
    @node avg = nsum(v3, v4, v5)   # sera divisé par 3 plus tard si nécessaire
end

# ── 3. Exécution forward ─────────────────────────────────────────────────
log = ExecutionLog()
demand!(g, :avg; ctx_store=CtxStore(), namespace=:vec, log=log)

# Afficher les valeurs finales
println("v0 = ", Array(g.nodes[:vec][:v0].value))
println("v1 = ", Array(g.nodes[:vec][:v1].value))
println("v5 = ", Array(g.nodes[:vec][:v5].value))
println("avg = ", Array(g.nodes[:vec][:avg].value))

# ── 4. Visualisation interactive ─────────────────────────────────────────
save_interactive_graph(g, log, "vec_stream.html"; title="Vector Stream")

In [ ]:
using NeuroDSL

# Opérateur générique : wsum avec un vecteur de poids
NeuroDSL.register_op!(:wsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
        w = attrs[:weights]
        @. out = w[1] * inputs[1]
        for i in 2:length(w)
            @. out += w[i] * inputs[i]
        end
    end
)
NeuroDSL.register_op!(:nsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
        copyto!(out, inputs[1])
        for i in 2:length(inputs)
            out .+= inputs[i]
        end
    end
)
NeuroDSL.register_op!(:identity,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> copyto!(out, inputs[1])
)

g = NeuroGraph(namespace=:vec, device=Backend.CPUDevice())

builder = @neuro g ns=:vec begin
    # Feuilles vectorielles nommées v0, v1, v2
    @node v0 = [1.0, 0.0, 0.0]
    @node v1 = [0.0, 1.0, 0.0]
    @node v2 = [0.0, 0.0, 1.0]

    # Règle Tribonacci vectorielle
    @rule tribo(n::Int) = n <= 2 ? Symbol(:v, n) : wsum(weights=[0.4, 0.3, 0.3], tribo(n-1), tribo(n-2), tribo(n-3))

    # Générer les termes suivants
    @node t3 = tribo(3)
    @node t4 = tribo(4)
    @node t5 = tribo(5)

    # Somme des trois
    @node total = nsum(t3, t4, t5)
end

log = ExecutionLog()
NeuroDSL.demand!(g, :total; ctx_store=CtxStore(), namespace=:vec, log=log)

println("t3 = ", Array(NeuroDSL.node(g, :t3; namespace=:vec).value))
println("t4 = ", Array(NeuroDSL.node(g, :t4; namespace=:vec).value))
println("t5 = ", Array(NeuroDSL.node(g, :t5; namespace=:vec).value))
println("total = ", Array(NeuroDSL.node(g, :total; namespace=:vec).value))

save_interactive_graph(g, log, "tribo_stream.html"; title="Tribonacci Vector Stream")

In [ ]:
# V2 tribo stream
using NeuroDSL

# Définition des opérateurs avec @defop
@defop wsum (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    w = attrs[:weights]
    @. out = w[1] * inputs[1]
    for i in 2:length(w)
        @. out += w[i] * inputs[i]
    end
end

@defop nsum (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
    for i in 2:length(inputs)
        out .+= inputs[i]
    end
end

@defop identity out = inputs[1]

# Construction du graphe
g = NeuroGraph(namespace=:vec, device=Backend.CPUDevice())

builder = @neuro g ns=:vec begin
    @node v0 = [1.0, 0.0, 0.0]
    @node v1 = [0.0, 1.0, 0.0]
    @node v2 = [0.0, 0.0, 1.0]

    @rule tribo(n::Int) = n <= 2 ? Symbol(:v, n) : wsum(weights=[0.4, 0.3, 0.3], tribo(n-1), tribo(n-2), tribo(n-3))

    @node t3 = tribo(3)
    @node t4 = tribo(4)
    @node t5 = tribo(5)

    @node total = nsum(t3, t4, t5)
end

# Exécution
log = ExecutionLog()
NeuroDSL.demand!(g, :total; ctx_store=CtxStore(), namespace=:vec, log=log)

println("t3 = ", Array(NeuroDSL.node(g, :t3; namespace=:vec).value))
println("t4 = ", Array(NeuroDSL.node(g, :t4; namespace=:vec).value))
println("t5 = ", Array(NeuroDSL.node(g, :t5; namespace=:vec).value))
println("total = ", Array(NeuroDSL.node(g, :total; namespace=:vec).value))

save_interactive_graph(g, log, "tribo_stream.html"; title="Tribonacci Vector Stream")

In [ ]:
# 1. Enregistrer les opérateurs personnalisés
NeuroDSL.register_op!(:wsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) ->
        (@. out = 0.3f0 * inputs[1] + 0.7f0 * inputs[2])
)

NeuroDSL.register_op!(:nsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
        copyto!(out, inputs[1])
        @inbounds for i in 2:length(inputs)
            out .+= inputs[i]
        end
    end
)

# 2. Construction du graphe Fibonacci
function build_fib_graph!(g, n; ns=:fib, vec_size=10)
    NeuroDSL.set!(g, :f0, randn(Float32, vec_size); namespace=ns)
    NeuroDSL.set!(g, :f1, randn(Float32, vec_size); namespace=ns)

    even_nodes = Symbol[:f0]        # f[0] est pair (indice 0)
    prev0, prev1 = :f0, :f1

    for i in 0:n-1
        f2 = Symbol(:f, i + 2)
        NeuroDSL.addrule!(g,
            NeuroDSL.GraphRule(f2, [prev0, prev1], :wsum; namespace=ns))
        prev0, prev1 = prev1, f2
        # i impair → f2 est à un indice pair
        i % 2 == 1 && push!(even_nodes, f2)
    end

    yn_sym = prev1      # dernier terme = f[n+1]

    # Nœud large : somme des termes pairs
    NeuroDSL.addrule!(g,
        NeuroDSL.GraphRule(:sn, even_nodes, :nsum; namespace=ns))

    return yn_sym, :sn
end

# 3. Exécution et benchmarking
n = 10
println("="^58)
@printf " NeuroDSL  Fibonacci DAG  n = %d\n" n
println("="^58)

t_build = @elapsed begin
    g = NeuroDSL.NeuroGraph(namespace=:fib, device=NeuroDSL.Backend.CPUDevice())
    yn_sym, sn_sym = build_fib_graph!(g, n; ns=:fib)
end
@printf "  Construction   %6d nœuds     %.4f s\n" (n+3) t_build

# Log d'exécution pour la visualisation
log = ExecutionLog()
ctx = CtxStore()

# Calcul de la somme des termes pairs (sn) et du dernier terme (yn)
NeuroDSL.invalidate_all!(g; namespace=:fib)
NeuroDSL.demand!(g, sn_sym; ctx_store=ctx, namespace=:fib, log=log)
NeuroDSL.demand!(g, yn_sym; ctx_store=ctx, namespace=:fib, log=log)

# Affichage des résultats
sn_val = Array(NeuroDSL.node(g, :sn; namespace=:fib).value)
yn_val = Array(NeuroDSL.node(g, yn_sym; namespace=:fib).value)
@printf "  sn (wide)  : [%.4f, %.4f, ...]\n" sn_val[1] sn_val[2]
@printf "  yn (deep)  : [%.4f, %.4f, ...]\n" yn_val[1] yn_val[2]

# Génération du fichier HTML interactif
save_interactive_graph(g, log, "fibonacci_n10.html"; title="Fibonacci DAG (n=10)")

println("\n✅ Visualisation sauvegardée dans fibonacci_n10.html")

In [ ]:
# une autre version de fibionacci
using NeuroDSL, Printf

# ── 1. Définition des opérateurs avec @defop ───────────────────────────
@defop wsum out = 0.3f0 * inputs[1] + 0.7f0 * inputs[2]

@defop nsum (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
    @inbounds for i in 2:length(inputs)
        out .+= inputs[i]
    end
end

# ── 2. Construction du graphe Fibonacci ────────────────────────────────
function build_fib_graph!(g, n; ns=:fib, vec_size=10)
    NeuroDSL.set!(g, :f0, randn(Float32, vec_size); namespace=ns)
    NeuroDSL.set!(g, :f1, randn(Float32, vec_size); namespace=ns)

    even_nodes = Symbol[:f0]        # f[0] est pair (indice 0)
    prev0, prev1 = :f0, :f1

    for i in 0:n-1
        f2 = Symbol(:f, i + 2)
        NeuroDSL.addrule!(g,
            NeuroDSL.GraphRule(f2, [prev0, prev1], :wsum; namespace=ns))
        prev0, prev1 = prev1, f2
        i % 2 == 1 && push!(even_nodes, f2)
    end

    yn_sym = prev1      # dernier terme = f[n+1]

    NeuroDSL.addrule!(g,
        NeuroDSL.GraphRule(:sn, even_nodes, :nsum; namespace=ns))

    return yn_sym, :sn
end

# ── 3. Exécution et benchmarking ───────────────────────────────────────
n = 10
println("="^58)
@printf " NeuroDSL  Fibonacci DAG  n = %d\n" n
println("="^58)

t_build = @elapsed begin
    g = NeuroDSL.NeuroGraph(namespace=:fib, device=NeuroDSL.Backend.CPUDevice())
    yn_sym, sn_sym = build_fib_graph!(g, n; ns=:fib)
end
@printf "  Construction   %6d nœuds     %.4f s\n" (n+3) t_build

log = ExecutionLog()
ctx = CtxStore()

NeuroDSL.invalidate_all!(g; namespace=:fib)
NeuroDSL.demand!(g, sn_sym; ctx_store=ctx, namespace=:fib, log=log)
NeuroDSL.demand!(g, yn_sym; ctx_store=ctx, namespace=:fib, log=log)

sn_val = Array(NeuroDSL.node(g, :sn; namespace=:fib).value)
yn_val = Array(NeuroDSL.node(g, yn_sym; namespace=:fib).value)
@printf "  sn (wide)  : [%.4f, %.4f, ...]\n" sn_val[1] sn_val[2]
@printf "  yn (deep)  : [%.4f, %.4f, ...]\n" yn_val[1] yn_val[2]

save_interactive_graph(g, log, "fibonacci_n10.html"; title="Fibonacci DAG (n=10)")

println("\n✅ Visualisation sauvegardée dans fibonacci_n10.html")

In [ ]:
# Supprimer le cache de précompilation
cache_dir = joinpath(homedir(), ".julia", "compiled", "v1.10", "NeuroDSL")
isdir(cache_dir) && rm(cache_dir; recursive=true, force=true)

# Charger le module avec include pour voir l'erreur entière
include(raw"C:\Users\Nevermind\Desktop\NeuroDSL\src\NeuroDSL.jl")

In [ ]:
using NeuroDSL, Printf

# ── 1. Enregistrement des opérateurs (version stable) ─────────────────────
NeuroDSL.register_op!(:wsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) ->
        (@. out = attrs[:weights][1] * inputs[1] + attrs[:weights][2] * inputs[2])
)
NeuroDSL.register_op!(:nsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
        copyto!(out, inputs[1])
        for i in 2:length(inputs)
            out .+= inputs[i]
        end
    end
)
NeuroDSL.register_op!(:identity,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> copyto!(out, inputs[1])
)

# ── 2. Construction du graphe déclaratif + liste dynamique externe ────────
function build_fib_graph(n::Int)
    g = NeuroGraph(namespace=:fib, device=Backend.CPUDevice())
    
    # Bloc @neuro (sans aucune compréhension de liste)
    builder = @neuro g ns=:fib begin
        @node f0 = randn(Float32, 10)
        @node f1 = randn(Float32, 10)
        @rule fib(n::Int) = n <= 1 ? Symbol(:f, n) : wsum(weights=[0.3, 0.7], fib(n-1), fib(n-2))
        @node yn = fib(n+1)
    end

    # Construction dynamique des termes pairs EN DEHORS de la macro
    even_terms = Symbol[call_rule(builder, :fib, i) for i in 0:1:n]   # Vector{Symbol} garanti
    addrule!(builder.graph, GraphRule(:sn, even_terms, :nsum; namespace=builder.namespace))

    return builder
end

# ── 3. Exécution forward ───────────────────────────────────────────────────
n = 10
builder = build_fib_graph(n)
g = builder.graph

log = ExecutionLog()
ctx = CtxStore()
NeuroDSL.demand!(g, :sn; ctx_store=ctx, namespace=:fib, log=log)
NeuroDSL.demand!(g, :yn; ctx_store=ctx, namespace=:fib, log=log)

sn_val = Array(NeuroDSL.node(g, :sn; namespace=:fib).value)
yn_val = Array(NeuroDSL.node(g, :yn; namespace=:fib).value)
@printf "  sn (wide) : [%.4f, %.4f, ...]\n" sn_val[1] sn_val[2]
@printf "  yn (deep) : [%.4f, %.4f, ...]\n" yn_val[1] yn_val[2]

# ── 4. Visualisation interactive ───────────────────────────────────────────
save_interactive_graph(g, log, "fibonacci_rec_n10.html"; title="Fibonacci Récursif (n=10)")
println("\n✅ Visualisation sauvegardée dans fibonacci_rec_n10.html")

In [ ]:
# V2 version fibionacci recursive
using NeuroDSL, Printf

# ── 1. Définition des opérateurs avec @defop ───────────────────────────
@defop wsum out = attrs[:weights][1] * inputs[1] + attrs[:weights][2] * inputs[2]

@defop nsum (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
    for i in 2:length(inputs)
        out .+= inputs[i]
    end
end

@defop identity out = inputs[1]

# ── 2. Construction du graphe déclaratif + liste dynamique externe ────────
function build_fib_graph(n::Int)
    g = NeuroGraph(namespace=:fib, device=Backend.CPUDevice())
    
    builder = @neuro g ns=:fib begin
        @node f0 = randn(Float32, 10)
        @node f1 = randn(Float32, 10)
        @rule fib(n::Int) = n <= 1 ? Symbol(:f, n) : wsum(weights=[0.3, 0.7], fib(n-1), fib(n-2))
        @node yn = fib(n+1)
    end

    even_terms = Symbol[call_rule(builder, :fib, i) for i in 0:1:n]
    addrule!(builder.graph, GraphRule(:sn, even_terms, :nsum; namespace=builder.namespace))

    return builder
end

# ── 3. Exécution forward ───────────────────────────────────────────────────
n = 10
builder = build_fib_graph(n)
g = builder.graph

log = ExecutionLog()
ctx = CtxStore()
NeuroDSL.demand!(g, :sn; ctx_store=ctx, namespace=:fib, log=log)
NeuroDSL.demand!(g, :yn; ctx_store=ctx, namespace=:fib, log=log)

sn_val = Array(NeuroDSL.node(g, :sn; namespace=:fib).value)
yn_val = Array(NeuroDSL.node(g, :yn; namespace=:fib).value)
@printf "  sn (wide) : [%.4f, %.4f, ...]\n" sn_val[1] sn_val[2]
@printf "  yn (deep) : [%.4f, %.4f, ...]\n" yn_val[1] yn_val[2]

save_interactive_graph(g, log, "fibonacci_rec_n10.html"; title="Fibonacci Récursif (n=10)")
println("\n✅ Visualisation sauvegardée dans fibonacci_rec_n10.html")

## 🔢 Binomial Coefficients – Odd Sum via Pascal’s Rule with NeuroDSL

### What this notebook does

We compute the sum of binomial coefficients $\displaystyle\sum_{k \text{ odd}} \binom{n}{k}$  using
a declarative, recursive approach powered by **NeuroDSL**. The computation is built
as a directed acyclic graph (DAG) where each \((n,k)\) pair is evaluated only once
thanks to automatic memoisation.

### Pascal’s rule

The binomial coefficients satisfy the recurrence

$$
\binom{n}{k} = \binom{n-1}{k-1} + \binom{n-1}{k},
$$

with boundary conditions  $ \binom{n}{0} = \binom{n}{n} = 1 $.

### NeuroDSL elements in this example

- **`@rule`** defines the recursive relationship (Pascal’s rule).
- **`wsum`** is a weighted sum operator registered with the `@defop` macro; here it
  simply adds the two recursive calls (weights \([1.0, 1.0]\)).
- **`nsum`** sums an arbitrary number of terms (used to accumulate the odd
  coefficients).
- **`@node`** declares constant values (the base case `one = 1.0`) and the final
  result `total`.
- **`@neuro`** builds the full recursive graph and handles the caching of
  intermediate results.

### Extensibility

The macro `@defop` lets you define custom operators in one line, making the DSL
easily adaptable to other recurrences (Fibonacci, Stirling, Tribonacci, etc.)
without touching the engine.

### Output

After execution, an interactive HTML trace is generated (`comb_odd_sum.html`),
allowing you to step through the computation forward pass and inspect every
intermediate value.

In [ ]:
using NeuroDSL, Printf

# ── 1. Opérateurs nécessaires ───────────────────────────────────────────
NeuroDSL.register_op!(:wsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) ->
        (@. out = attrs[:weights][1] * inputs[1] + attrs[:weights][2] * inputs[2])
)
NeuroDSL.register_op!(:nsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
        copyto!(out, inputs[1])
        for i in 2:length(inputs)
            out .+= inputs[i]
        end
    end
)
NeuroDSL.register_op!(:identity,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> copyto!(out, inputs[1])
)

# ── 2. Graphe déclaratif ────────────────────────────────────────────────
n = 6
g = NeuroGraph(namespace=:comb, device=Backend.CPUDevice())

builder = @neuro g ns=:comb begin
    # Cas de base C(n,0)=1, C(n,n)=1
    @node one = 1.0

    # Règle récursive de Pascal
    @rule comb(n::Int, k::Int) = (k == 0 || k == n) ? :one :
                                 wsum(weights=[1.0, 1.0], comb(n-1, k-1), comb(n-1, k))

    # Somme sur k pair de 0 à n
    even_ks = [comb(n, k) for k in 1:2:n]
    @node total = nsum(even_ks...)
end

# ── 3. Exécution forward ────────────────────────────────────────────────
log = ExecutionLog()
ctx = CtxStore()
NeuroDSL.demand!(g, :total; ctx_store=ctx, namespace=:comb, log=log)

# ── 4. Affichage du résultat ────────────────────────────────────────────
val = Array(NeuroDSL.node(g, :total; namespace=:comb).value)
println("Somme des C(10, k) pour k pair = $(val[])")   # 512.0 (c'est 2^(n-1) = 512)

# ── 5. Visualisation interactive ────────────────────────────────────────
save_interactive_graph(g, log, "comb_even_sum.html"; title="Somme C(10,k) pour k pair")

In [2]:
# V2 version de combinaison 
using NeuroDSL, Printf

# ── 1. Définition des opérateurs avec @defop ───────────────────────────
@defop wsum out = attrs[:weights][1] * inputs[1] + attrs[:weights][2] * inputs[2]

@defop nsum (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
    for i in 2:length(inputs)
        out .+= inputs[i]
    end
end

@defop identity out = inputs[1]

# ── 2. Graphe déclaratif ────────────────────────────────────────────────
n = 6
g = NeuroGraph(namespace=:comb, device=Backend.CPUDevice())

builder = @neuro g ns=:comb begin
    @node one = 1.0
    @rule comb(n::Int, k::Int) = (k == 0 || k == n) ? :one :
                                 wsum(weights=[1.0, 1.0], comb(n-1, k-1), comb(n-1, k))
    even_ks = [comb(n, k) for k in 1:2:n]
    @node total = nsum(even_ks...)
end

# ── 3. Exécution forward ────────────────────────────────────────────────
log = ExecutionLog()
ctx = CtxStore()
NeuroDSL.demand!(g, :total; ctx_store=ctx, namespace=:comb, log=log)

val = Array(NeuroDSL.node(g, :total; namespace=:comb).value)
println("Somme des C($n, k) pour k impair = $(val[])")

save_interactive_graph(g, log, "comb_odd_sum.html"; title="Somme C($n,k) pour k impair")

✅ Op :wsum registered
✅ Op :nsum registered
✅ Op :identity registered
Somme des C(6, k) pour k impair = 32.0
✅ Interactive Trace exported → comb_odd_sum.html


In [ ]:
using NeuroDSL, Printf

NeuroDSL.register_op!(:scale_add,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
        factor = attrs[:factor]
        @. out = factor * inputs[1] + inputs[2]
    end
)

NeuroDSL.register_op!(:nsum,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
        copyto!(out, inputs[1])
        for i in 2:length(inputs)
            out .+= inputs[i]
        end
    end
)

NeuroDSL.register_op!(:identity,
    (dev, out, inputs, attrs, out_sym, nd, ctx) -> copyto!(out, inputs[1])
)
g = NeuroGraph(namespace=:stirling, device=Backend.CPUDevice())

builder = @neuro g ns=:stirling operators=[:scale_add] begin
    @node one  = 1.0
    @node zero = 0.0

    @rule stirling(n::Int, k::Int) = begin
        if n == 0 && k == 0
            :one
        elseif k == 0 || k > n
            :zero
        else
            scale_add(factor=k, stirling(n-1, k), stirling(n-1, k-1))
        end
    end

    @node s53 = stirling(5, 3)
    all_terms = [stirling(5, k) for k in 1:5]
    @node bell5 = nsum(all_terms...)
end


# ── 3. Exécution ───────────────────────────────────────────────────────────
log = ExecutionLog()
ctx = CtxStore()
NeuroDSL.demand!(g, :s53; ctx_store=ctx, namespace=:stirling, log=log)
NeuroDSL.demand!(g, :bell5; ctx_store=ctx, namespace=:stirling, log=log)

s53_val = Array(NeuroDSL.node(g, :s53; namespace=:stirling).value)[]
bell5_val = Array(NeuroDSL.node(g, :bell5; namespace=:stirling).value)[]
println("S(5,3) = $(s53_val)")   # doit être 25
println("Somme S(5,k) pour k=1..5 = $(bell5_val) (B₅ = 52)")

# ── 4. Visualisation ───────────────────────────────────────────────────────
save_interactive_graph(g, log, "stirling_example.html"; title="Nombres de Stirling S(5,3)")
println("✅ Visualisation sauvegardée dans stirling_example.html")

## 🔢 Stirling Numbers of the Second Kind – Recursive Computation with NeuroDSL

### Mathematical Definition

The **Stirling numbers of the second kind**, denoted $S(n,k)$ or $\left\{ {n \atop k} \right\}$, 
count the number of ways to partition a set of $n$ elements into $k$ non‑empty, disjoint subsets. 
They satisfy the recurrence:

$$
S(n,k) =
\begin{cases}
1 & \text{if } n = 0 \text{ and } k = 0,\\
0 & \text{if } k = 0 \text{ or } k > n,\\
k \cdot S(n-1,k) + S(n-1,k-1) & \text{otherwise.}
\end{cases}
$$

The sum of $S(n,k)$ for $1 \le k \le n$ yields the **Bell number** $B_n$, which gives the 
total number of partitions of an $n$-element set.

---

### NeuroDSL Connection

This notebook uses **NeuroDSL**, a Julia DSL for building and executing recursive computation 
graphs (DAGs) in a declarative way.

- **`@rule`** defines the Stirling recurrence.
- **`scale_add`** is a custom operator that computes $k \times A + B$, registered with the 
  `@defop` macro.
- **`@node`** declares constant leaves or intermediate results.
- **`@neuro`** builds the recursive graph with **automatic memoisation** – each pair $(n,k)$ 
  is computed exactly once.
- The final sum `bell5` is obtained via the `nsum` operator (element‑wise sum of vectors).

**Extensibility**: with `@defop`, you can define your own operators in a single line, without 
modifying the engine. Example:
```julia
@defop scale_add out = attrs[:factor] * inputs[1] + inputs[2]
```

In [3]:
# Stirling Number V2
using NeuroDSL, Printf

# Operator definitions with @defop
@defop scale_add out = attrs[:factor] * inputs[1] + inputs[2]

@defop nsum (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
    for i in 2:length(inputs)
        out .+= inputs[i]
    end
end

@defop identity out = inputs[1]

# Parameterized Stirling graph construction
function build_stirling_graph(g, n, k)
    builder = @neuro g ns=:stirling operators=[:scale_add] begin
        @node one  = 1.0
        @node zero = 0.0

        @rule stirling(n::Int, k::Int) = begin
            if n == 0 && k == 0
                :one
            elseif k == 0 || k > n
                :zero
            else
                scale_add(factor=k, stirling(n-1, k), stirling(n-1, k-1))
            end
        end

        # n and k are captured by the enclosing function's closure
        @node s53 = stirling(n, k)               # S(5,3)
        all_terms = [stirling(n, i) for i in 1:n] # all S(5,i)
        @node bell5 = nsum(all_terms...)          # sum = B₅
    end
    return builder
end

# Usage
n, k = 5, 3
g = NeuroGraph(namespace=:stirling, device=Backend.CPUDevice())
builder = build_stirling_graph(g, n, k)

log = ExecutionLog()
ctx = CtxStore()
NeuroDSL.demand!(g, :s53;   ctx_store=ctx, namespace=:stirling, log=log)
NeuroDSL.demand!(g, :bell5; ctx_store=ctx, namespace=:stirling, log=log)

s53_val   = Array(NeuroDSL.node(g, :s53;   namespace=:stirling).value)[]
bell5_val = Array(NeuroDSL.node(g, :bell5; namespace=:stirling).value)[]
println("S($n,$k) = $(s53_val)")                     # 25.0
println("Sum S($n,k) for k=1..$n = $(bell5_val) (B₅ = 52)")

save_interactive_graph(g, log, "stirling_example.html"; title="Stirling numbers S($n,$k)")
println("✅ Visualization saved to stirling_example.html")

✅ Op :scale_add registered
✅ Op :nsum registered
✅ Op :identity registered
S(5,3) = 25.0
Sum S(5,k) for k=1..5 = 52.0 (B₅ = 52)
✅ Interactive Trace exported → stirling_example.html
✅ Visualization saved to stirling_example.html


In [4]:
using NeuroDSL, Printf

# Opérateurs définis avec la macro @defop
@defop scale_add out = attrs[:factor] * inputs[1] + inputs[2]
@defop identity  out = inputs[1]
@defop nsum (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
    for i in 2:length(inputs)
        out .+= inputs[i]
    end
end

function build_stirling_graph(g, n)
    builder = @neuro g ns=:stirling operators=[:scale_add] begin
        @node one  = 1.0
        @node zero = 0.0

        @rule stirling(n::Int, k::Int) = begin
            if n == 0 && k == 0
                :one
            elseif k == 0 || k > n
                :zero
            else
                scale_add(factor=k, stirling(n-1, k), stirling(n-1, k-1))
            end
        end

        all_terms = [stirling(n, i) for i in 1:n]
        @node bell = nsum(all_terms...)
    end
    return builder
end

n = 6   # B₆ = 203
g = NeuroGraph(namespace=:stirling, device=Backend.CPUDevice())
builder = build_stirling_graph(g, n)

log = ExecutionLog()
ctx = CtxStore()
NeuroDSL.demand!(g, :bell; ctx_store=ctx, namespace=:stirling, log=log)

bell_val = Array(NeuroDSL.node(g, :bell; namespace=:stirling).value)[]
println("B_$n = $bell_val")

save_interactive_graph(g, log, "stirling_bell_6.html"; title="Stirling & Bell numbers (n=$n)")

✅ Op :scale_add registered
✅ Op :identity registered
✅ Op :nsum registered
B_6 = 203.0
✅ Interactive Trace exported → stirling_bell_6.html
